# Practice guide 04_02: dynamic analysis — sandbox API call sequences

**Goal.** Work the sequential side of malware detection yourself: build the bag-of-calls features from raw
API ids, train the GRU, run the **order-matters** experiment that decides between the two, and read the
per-family errors of a 7-class sequence model. Every `TODO` cell must be completed; the `check` cells tell
you whether you got it right.

Read `guide_04_02.pdf` first. Time: about 2 hours. Cells marked **(given)** are complete.

**Environment.** `make venv` at the repository root; Keras runs on the JAX backend (no GPU needed).

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")

import sys
from pathlib import Path

import numpy as np
from scipy import sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

LIB = next(p for p in (Path("../../notebooks/03-malware"), Path("../../../notebooks/03-malware")) if p.exists())
sys.path.insert(0, str(LIB))  # student / solution folder
import malwlib as mw

## Part A. The sequences (given)

We work on a synthetic stand-in: 4 behaviour profiles with distinct API subranges and burst structure,
4,000 files x 100 calls, 4 classes (0 = benign, 1..3 families), ids 1..306 (0 = padding). At 100 integers
per file this is ~1.6 MB dense — laptop-safe. The real Cuckoo sequences of oliveira et al. (2019) are
43,876 x 100, 97.5% malware, and load with mw.load_sequences() — the mirror image: there the *benign*
class is the rare one (as in guide 04_01).

In [ ]:
Xs, ys = mw.synth_sequences()  # 4 classes: 0=benign, 1..3 families; mw.load_sequences() for the real data
print(f"sequences: {Xs.shape[0]:,} x {Xs.shape[1]}, vocab {Xs.max() + 1}, benign {1 - ys.mean():.2%}")
Str, Ste, ytr, yte = train_test_split(Xs, ys, test_size=0.25, stratify=ys, random_state=0)

**Question A1.** In the real dataset 97.5% of files are malware. What is the accuracy of a model that
predicts *malware* for everything, and which two metrics expose it? Which class do you expect the per-class
F1 of a real model to suffer on?

**Answers A1.** Constant-malware accuracy is 0.975 — and it is worthless. Recall is 1.0 but precision equals
the base rate 0.975 only if there are no false positives, i.e. the pair (precision, recall) hides nothing:
what exposes the trivial model is the *benign-class* recall (0) and the PR curve for the benign class, whose
average precision equals the base rate for a random score. Any real model will have its weakest per-class F1
on benign (1,079 files) — exactly where we most need it to be right.

## Part B. Bag of calls from raw ids

Build the order-free features yourself: for each file, a vector over the 307-API vocabulary counting the
occurrences of each call id (padding id 0 excluded). Return a scipy CSR — dense would be 43876 x 307 x 8
bytes when it does not need to be.

In [ ]:
def bag_of_calls(S, vocab=None):
    """S (n, 100) int API ids -> CSR (n, vocab) float counts; padding id 0 dropped."""
    vocab = vocab or int(S.max()) + 1
    rows = np.repeat(np.arange(len(S)), S.shape[1])
    mask = S.ravel() > 0
    return sp.csr_matrix((np.ones(mask.sum()), (rows[mask], S.ravel()[mask])), shape=(len(S), vocab))

In [ ]:
# check: shape, dtype, and the count of a known id
Btr, Bte = bag_of_calls(Str), bag_of_calls(Ste)
assert Btr.shape == (len(Str), int(Xs.max()) + 1) and Bte.shape == (len(Ste), int(Xs.max()) + 1)
i0 = np.where(Str[0] > 0)[0]
call_id = Str[0, i0[0]]
assert Btr[0, call_id] == int((Str[0] == call_id).sum())  # count matches raw
pairs = {(i, c) for i in range(len(Str)) for c in set(Str[i]) if c > 0}
assert Btr.nnz == len(pairs)  # one entry per distinct (row, call id) pair
print(f"bag-of-calls: {Btr.shape}, nnz {Btr.nnz:,}, ~{Btr.data.nbytes / 1e6:.1f} MB of data")

**Question B1.** Why is the sparse matrix the right container here — give the density that makes CSR pay
off against a dense float32 array.

**Answers B1.** A file makes at most 100 distinct calls out of 307 ids, and after padding usually far fewer:
the density is about (calls)/(n x 307), of order 20% for the padded matrix and below 1% if the vocabulary
grows to thousands of APIs. CSR pays whenever nnz x 12 bytes (value + index) is much smaller than n x d x 4:
already at ~25% density for int32 data, and overwhelmingly so for the real 21,918-column fingerprint of
guide 04_01 (0.3%).

In [ ]:
lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=0).fit(Btr, ytr)
pb = lr.predict_proba(Bte)[:, 1]  # P(malware): class 0 is benign, classes 1..3 are families
mw.eval_bin(yte > 0, pb, "bag-of-calls (malware vs benign): ")

## Part C. The GRU

The model of the lecture: `Embedding(vocab, 32, mask_zero=True) -> GRU(64) -> Dense(2)`. Train it 3 epochs,
batch 128, validation split 0.2, and report malware-vs-benign AUC/F1 with `mw.eval_bin(yte > 0, pg)` —
the softmax's second column is P(malware).

In [ ]:
import keras
from keras import layers

keras.utils.set_random_seed(0)
model = keras.Sequential([
    layers.Input((100,)),
    layers.Embedding(int(Xs.max()) + 1, 32, mask_zero=True),
    layers.GRU(64),
    layers.Dense(2, activation="softmax"),
])
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
hist = model.fit(Str, ytr, epochs=3, batch_size=128, validation_split=0.2, verbose=0)
pg = model.predict(Ste, verbose=0)[:, 1]  # (n, 2): column 1 is P(malware)
mw.eval_bin(yte > 0, pg, "GRU (malware vs benign): ")
print(f"bag-of-calls F1 {f1_score(yte > 0, pb > .5):.4f} vs GRU F1 {f1_score(yte > 0, pg > .5):.4f}")

**Question C1.** The GRU wins on F1 but can lose on AUC. Explain why a recurrent model trained on 97.5%
malware tends to push scores to the extremes, and what the validation loss tells you about 3 epochs.

**Answers C1.** Cross-entropy on an imbalanced set is minimised by confident predictions: the model learns
that *malware* is almost always right, so its probabilities saturate near 0 and 1, which helps F1 at a fixed
0.5 threshold but adds noise to the ranking (AUC). The validation loss plateaus (or rises) after the second
epoch while the training loss keeps falling: 3 epochs are enough, more would overfit the frequent families.

## Part D. Order matters

The decisive experiment: shuffle the 100 calls *within each training file* (permutation along axis 1, same
multiset), retrain the same GRU, compare. Implement the shuffle and the retrain.

In [ ]:
rng = np.random.default_rng(0)
Ssh = rng.permuted(Str, axis=1)
keras.utils.set_random_seed(0)
model2 = keras.Sequential([
    layers.Input((100,)),
    layers.Embedding(int(Xs.max()) + 1, 32, mask_zero=True),
    layers.GRU(64),
    layers.Dense(2, activation="softmax"),
])
model2.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
model2.fit(Ssh, ytr, epochs=3, batch_size=128, validation_split=0.2, verbose=0)
pg_sh = model2.predict(Ste, verbose=0)[:, 1]
full = mw.eval_bin(yte > 0, pg, "full: ")
shuf = mw.eval_bin(yte > 0, pg_sh, "shuffled: ")
order_gap = (
    f1_score(yte > 0, pg > 0.5) - f1_score(yte > 0, pg_sh > 0.5),
    full["auc"] - shuf["auc"],
)
print(f"order gap: dF1 {order_gap[0]:+.4f}, dAUC {order_gap[1]:+.4f}")

**Question D1.** Interpret your gaps. If the shuffled model matched the full one, what would that say about
the dataset — and about the bag-of-calls baseline of part B?

**Answers D1.** A clear positive gap means the GRU really reads order: the shuffled model only sees the same
multiset, so anything it still achieves is multiset information. A near-zero gap would mean the families
separate by *which* calls they make, not in what order — and the bag-of-calls baseline was the right model
all along (cheaper, sparse, interpretable). On this dataset the F1 gap is about -0.002 (shuffled is even slightly higher) and only the AUC gap is
positive (about 0.04): order carries a small ranking signal, and the bag of calls is nearly as good.

## Part E. Families from sequences

Train the bag-of-calls logistic regression on the 6 malware families only (binary benign class dropped) and
print the classification report. Which family is hardest, and is that a property of the *calls* it makes?

In [ ]:
fam_tr, fam_te = ytr[ytr > 0] - 1, yte[yte > 0] - 1  # families 1..3 -> 0..2
Btr_m, Bte_m = bag_of_calls(Str[ytr > 0]), bag_of_calls(Ste[yte > 0])
clf3 = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=0).fit(Btr_m, fam_tr)
pred3 = clf3.predict(Bte_m)
print(classification_report(fam_te, pred3, labels=[0, 1, 2], digits=3, zero_division=0,
                            target_names=mw.FAMILIES[1:3]))

**Question E1.** The weakest family makes few, generic calls (file opens, registry reads) that benign
software also makes. Why does `class_weight="balanced"` help its recall but not its precision?

**Answers E1.** Balanced weights multiply the rare family's loss, so the model moves its boundary to catch
more of it (recall up). But its calls overlap the other classes' — the *same* features point at several
families — so the extra catches include files of other families: precision cannot rise without a feature
that separates them. More data or features (order, call arguments, n-grams of calls) is the only way.

# Reference solution

The solved notebook, with the answers to the questions and the numbers of a full run, is `solution/guide_04_02.ipynb`.